# Popolamento MongoDB — Collezione `recordings`
Script autonomo da eseguire **una sola volta** su Kaggle.

**Cosa fa:**
1. Legge i secret Kaggle (`MONGO_URI`, `GDRIVE_API_KEY`)
2. Si connette a MongoDB Atlas
3. Scansiona le cartelle Google Drive (una per partecipante)
4. Per ogni coppia `.wav` + `.json`, inserisce un documento in `recordings`
5. Crea indici utili per le query della pipeline principale

In [2]:
# ── Dipendenze ────────────────────────────────────────────────
!pip install -q pymongo google-api-python-client

In [3]:
import io
import json

from kaggle_secrets import UserSecretsClient
from pymongo import MongoClient, ASCENDING
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

print('Librerie importate')

Librerie importate


In [4]:
# ══════════════════════════════════════════════════════════════
# CONFIGURAZIONE — modifica solo questi valori se necessario
# ══════════════════════════════════════════════════════════════

# ID della cartella radice su Google Drive
# (la cartella che contiene le sottocartelle dei partecipanti)
FOLDER_ID = "1KyAc_ke41rdGeAlJCtgG2TlFBL9DB8KU"

# Nome del database e della collezione su MongoDB Atlas
DB_NAME          = "asr_dialects_db_v3"
COLLECTION_NAME  = "recordings"

# Nomi dei secret Kaggle
SECRET_MONGO     = "MONGO_URI"
SECRET_GDRIVE    = "GDRIVE_API_KEY"

print('Configurazione caricata')

Configurazione caricata


In [5]:
# ══════════════════════════════════════════════════════════════
# STEP 1 — Lettura secret Kaggle
# ══════════════════════════════════════════════════════════════

secrets    = UserSecretsClient()
MONGO_URI  = secrets.get_secret(SECRET_MONGO)
GDRIVE_KEY = secrets.get_secret(SECRET_GDRIVE)

print('✅ Secret letti correttamente')

✅ Secret letti correttamente


In [6]:
# ══════════════════════════════════════════════════════════════
# STEP 2 — Connessione MongoDB Atlas
# ══════════════════════════════════════════════════════════════

mongo_client   = MongoClient(MONGO_URI)
db             = mongo_client[DB_NAME]
recordings_col = db[COLLECTION_NAME]

# Verifica connessione
mongo_client.admin.command('ping')
print(f'✅ Connesso a MongoDB Atlas — db: {DB_NAME}')
print(f'   Documenti già presenti in recordings: {recordings_col.count_documents({})}')

✅ Connesso a MongoDB Atlas — db: asr_dialects_db_v3
   Documenti già presenti in recordings: 0


In [7]:
# ══════════════════════════════════════════════════════════════
# STEP 3 — Connessione Google Drive
# ══════════════════════════════════════════════════════════════

drive_service = build('drive', 'v3', developerKey=GDRIVE_KEY)
print('✅ Google Drive API pronta')

✅ Google Drive API pronta


In [10]:
# ══════════════════════════════════════════════════════════════
# STEP 4 — Funzioni di utilità Drive
# ══════════════════════════════════════════════════════════════

def get_subfolders(folder_id: str) -> list:
    """
    Restituisce le sottocartelle dirette di una cartella Drive.
    Ogni sottocartella = un partecipante.
    """
    items      = []
    page_token = None
    while True:
        resp = drive_service.files().list(
            q=(
                f"'{folder_id}' in parents "
                f"and trashed=false "
                f"and mimeType='application/vnd.google-apps.folder'"
            ),
            fields="nextPageToken, files(id, name)",
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        items.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return items


def get_folder_files(folder_id: str) -> list:
    """
    Restituisce tutti i file (non cartelle) dentro una cartella Drive.
    """
    items      = []
    page_token = None
    while True:
        resp = drive_service.files().list(
            q=(
                f"'{folder_id}' in parents "
                f"and trashed=false "
                f"and mimeType!='application/vnd.google-apps.folder'"
            ),
            fields="nextPageToken, files(id, name, mimeType, size)",
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        items.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return items


def download_json_in_memory(file_id: str) -> dict:
    """
    Scarica un file JSON da Drive direttamente in memoria
    senza scriverlo su disco.
    """
    request    = drive_service.files().get_media(fileId=file_id)
    buffer     = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done       = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    return json.loads(buffer.read().decode('utf-8'))


print('✅ Funzioni Drive definite')

✅ Funzioni Drive definite


In [11]:
# ══════════════════════════════════════════════════════════════
# STEP 5 — Scansione Drive e popolamento MongoDB
# ══════════════════════════════════════════════════════════════

n_existing = recordings_col.count_documents({})

if n_existing > 0:
    print(f'⏭️  La collezione recordings è già popolata ({n_existing} documenti) — skip.')
    print('   Per re-indicizzare: recordings_col.delete_many({}) e riesegui.')

else:
    print(f'🔍 Scansione cartella Drive: {FOLDER_ID}')
    subfolders = get_subfolders(FOLDER_ID)
    print(f'   Trovate {len(subfolders)} sottocartelle (partecipanti)\n')

    docs_to_insert = []
    warnings       = []

    for folder in subfolders:
        folder_id   = folder['id']
        folder_name = folder['name']   # es. "2026-04-21_09-28-17_napoletano_F_50-59"

        files = get_folder_files(folder_id)

        # Indicizza per basename (nome senza estensione)
        wav_map  = {
            f['name'].rsplit('.', 1)[0]: f
            for f in files if f['name'].lower().endswith('.wav')
        }
        json_map = {
            f['name'].rsplit('.', 1)[0]: f
            for f in files if f['name'].lower().endswith('.json')
        }

        print(f'  📁 {folder_name}: {len(wav_map)} wav, {len(json_map)} json')

        for basename, wav_file in wav_map.items():

            # ── Caso 1: JSON presente → leggi i metadati da lì ──
            if basename in json_map:
                try:
                    meta = download_json_in_memory(json_map[basename]['id'])
                except Exception as e:
                    msg = f'Errore lettura JSON per {basename}: {e}'
                    print(f'     ❌ {msg}')
                    warnings.append(msg)
                    continue

            # ── Caso 2: JSON mancante → metadati minimi dal nome file ──
            else:
                msg = f'JSON mancante per {wav_file["name"]} in {folder_name} — metadati minimi'
                print(f'     ⚠️  {msg}')
                warnings.append(msg)
                meta = {
                    'participantId':  folder_name,
                    'filename':       wav_file['name'],
                    'promptId':       None,
                    'promptText':     None,
                    'promptCategory': None,
                    'recordingIndex': None,
                    'size':           int(wav_file.get('size', 0)),
                    'mimetype':       wav_file.get('mimeType', 'audio/wav'),
                    'uploadedAt':     None,
                }

            # ── Costruisce il documento MongoDB ──────────────────
            # _id univoco: participantId + filename
            doc_id = f"{meta.get('participantId', folder_name)}__{meta.get('filename', wav_file['name'])}"

            doc = {
                '_id':             doc_id,
                'participantId':   meta.get('participantId',  folder_name),
                'promptId':        meta.get('promptId'),
                'promptText':      meta.get('promptText'),
                'promptCategory':  meta.get('promptCategory'),
                'recordingIndex':  meta.get('recordingIndex'),
                'filename':        meta.get('filename',       wav_file['name']),
                'size':            meta.get('size'),
                'mimetype':        meta.get('mimetype',       'audio/wav'),
                'uploadedAt':      meta.get('uploadedAt'),
                # Campi Drive — già disponibili, non serve aggiornamento successivo
                'drive_file_id':   wav_file['id'],
                'drive_folder_id': folder_id,
            }
            docs_to_insert.append(doc)

    # ── Inserimento bulk ──────────────────────────────────────
    if docs_to_insert:
        # ordered=False: continua anche se qualche _id è duplicato
        result = recordings_col.insert_many(docs_to_insert, ordered=False)
        print(f'\n✅ Inseriti {len(result.inserted_ids)} documenti in recordings')
    else:
        print('\n⚠️  Nessun documento da inserire')

    if warnings:
        print(f'\n⚠️  Avvisi ({len(warnings)}):')
        for w in warnings:
            print(f'   - {w}')

    # ── Verifica finale ───────────────────────────────────────
    total = recordings_col.count_documents({})
    print(f'\n📊 Totale documenti in recordings: {total}')
    esempio = recordings_col.find_one({}, {'promptText': 0, 'drive_file_id': 0})
    print(f'Esempio documento:\n{json.dumps(esempio, indent=2, default=str)}')

🔍 Scansione cartella Drive: 1KyAc_ke41rdGeAlJCtgG2TlFBL9DB8KU
   Trovate 29 sottocartelle (partecipanti)

  📁 2026-04-22_12-03-17_altro_F_60-69: 1 wav, 1 json
  📁 2026-04-21_09-28-17_napoletano_F_50-59: 1 wav, 1 json
  📁 2026-04-21_20-34-47_napoletano_M_50-59: 1 wav, 1 json
  📁 2026-04-22_12-03-32_napoletano_F_30-39: 1 wav, 1 json
  📁 2026-04-22_14-09-28_napoletano_M_50-59: 1 wav, 1 json
  📁 2026-04-22_12-27-51_napoletano_F_50-59: 1 wav, 1 json
  📁 2026-04-22_12-02-47_altro_M_60-69: 1 wav, 1 json
  📁 2026-04-22_14-28-08_napoletano_M_50-59: 1 wav, 1 json
  📁 2026-04-22_14-03-52_napoletano_M_50-59: 1 wav, 1 json
  📁 2026-04-22_12-57-30_napoletano_M_50-59: 1 wav, 1 json
  📁 2026-04-22_13-23-21_napoletano_M_80plus: 1 wav, 1 json
  📁 2026-04-22_17-00-13_napoletano_F_30-39: 1 wav, 1 json
  📁 2026-04-22_17-04-10_napoletano_M_40-49: 3 wav, 3 json
  📁 2026-04-22_17-33-53_napoletano_F_18-29: 1 wav, 1 json
  📁 2026-04-23_18-41-20_napoletano_F_50-59: 2 wav, 2 json
  📁 2026-04-22_18-58-27_napoletan

In [12]:
# ══════════════════════════════════════════════════════════════
# STEP 6 — Creazione indici MongoDB (eseguibile anche più volte)
# ══════════════════════════════════════════════════════════════

recordings_col.create_index([('participantId',  ASCENDING)], name='idx_participantId')
recordings_col.create_index([('promptId',       ASCENDING)], name='idx_promptId')
recordings_col.create_index([('promptCategory', ASCENDING)], name='idx_promptCategory')
recordings_col.create_index([('filename',       ASCENDING)], name='idx_filename', unique=True)

print('✅ Indici creati:')
for idx in recordings_col.list_indexes():
    print(f'   - {idx["name"]}: {idx["key"]}')

✅ Indici creati:
   - _id_: SON([('_id', 1)])
   - idx_participantId: SON([('participantId', 1)])
   - idx_promptId: SON([('promptId', 1)])
   - idx_promptCategory: SON([('promptCategory', 1)])
   - idx_filename: SON([('filename', 1)])


In [15]:
from pathlib import Path
DATASET_PATH = "/kaggle/input/datasets/andrearoscigno/bigdata"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

# Collezione su MongoDB (mantenuta solo quella dei partecipanti)
participants_col = db["participants"]

print(f"Connesso con successo al database: {DB_NAME}")

# ==========================================
# 2. CARICAMENTO METADATI PARTECIPANTI
# ==========================================
participants_dir = Path(DATASET_PATH) / "participants"
participants_docs = []

print("\n--- Inizio caricamento partecipanti ---")
if participants_dir.exists():
    for json_file in participants_dir.glob("*.json"):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # Usiamo il participantId come chiave primaria (_id) per evitare duplicati
                if "participantId" in data:
                    data["_id"] = data["participantId"]
                
                participants_docs.append(data)
        except Exception as e:
            print(f"Errore nella lettura del file {json_file.name}: {e}")

    if participants_docs:
        # L'operazione 'upsert' evita errori se esegui lo script più volte
        for doc in participants_docs:
            participants_col.update_one({"_id": doc["_id"]}, {"$set": doc}, upsert=True)
        print(f"Caricati/Aggiornati {len(participants_docs)} profili partecipante.")
else:
    print(f"Errore: La cartella {participants_dir} non esiste.")

print("\nProcesso completato!")

Connesso con successo al database: asr_dialects_db_v3

--- Inizio caricamento partecipanti ---
Caricati/Aggiornati 29 profili partecipante.

Processo completato!
